In [71]:
# Install necessary packages
!pip install cerebras_cloud_sdk networkx matplotlib
!pip install gradio requests python-dotenv

# Import the Cerebras SDK for interacting with the Cerebras cloud services
from cerebras.cloud.sdk import Cerebras

In [72]:
# Import necessary libraries
import os  # For interacting with the operating system and environment variables
import gradio as gr  # For creating web-based user interfaces
from dataclasses import dataclass  # For creating data classes
import ast  # For parsing Python syntax trees
import asyncio  # For asynchronous programming
from typing import Dict, List, Optional  # For type hinting
from dotenv import load_dotenv  # For loading environment variables from a .env file

# Load environment variables from a .env file
load_dotenv()

# Set the Cerebras API key as an environment variable
# (Note: You should never hardcode sensitive information like API keys in your code)
os.environ["CEREBRAS_API_KEY"] = "csk-9n8rvhn6ydpkrfxf6fwhdptpfpd5exyje4cy3tyrkm3m8txv"

In [73]:
@dataclass
class SecurityIssue:
    """Represents a security issue found in the analyzed code."""
    severity: str  # Severity level (High, Medium, Low)
    description: str  # Description of the issue
    line_number: int  # Line number where the issue was found
    suggestion: str  # Suggested fix for the issue
    code_snippet: str  # Code snippet related to the issue

class CodeSecurityAnalyzer:
    """Analyzes Python code for security vulnerabilities using the Cerebras API."""

    def __init__(self):
        # Initialize the analyzer with the Cerebras API client, using an API key from environment variables
        self.client = Cerebras(
            api_key=os.environ.get("CEREBRAS_API_KEY")
        )

    async def analyze_code_snippet(self, code: str) -> List[SecurityIssue]:
        """Analyzes a given code snippet for security vulnerabilities."""

        # Create a prompt for the API to analyze the code
        prompt = f"""You are a senior security engineer. Analyze the following code for security vulnerabilities.
        Focus on:
        1. SQL Injection
        2. XSS vulnerabilities
        3. Command Injection
        4. Hardcoded secrets
        5. Unsafe deserialization
        6. Path traversal

        For each vulnerability found, provide:
        - Severity (High/Medium/Low)
        - Description
        - Line number
        - Secure alternative

        Code to analyze:
        ```python
        {code}
        ```

        Format your response as a JSON list of issues. Be precise and concise."""

        try:
            # Send the prompt to the Cerebras API and get the analysis result
            chat_completion = self.client.chat.completions.create(
                messages=[
                    {"role": "user", "content": prompt}
                ],
                model="llama3.1-70b",
                temperature=0.1,
                max_tokens=1000
            )

            # Extract the response content
            analysis = chat_completion.choices[0].message.content
            # Parse the analysis response into a list of SecurityIssue objects
            return self._parse_security_analysis(analysis, code)
        except Exception as e:
            # Handle any exceptions that occur during the analysis
            return [SecurityIssue(
                severity="Error",
                description=f"Analysis failed: {str(e)}",
                line_number=0,
                suggestion="Please try again",
                code_snippet=""
            )]

    def _parse_security_analysis(self, analysis_text: str, original_code: str) -> List[SecurityIssue]:
        """Parses the API response and performs static analysis on the code."""
        issues = []

        # Perform static analysis to find potential issues in the code
        try:
            tree = ast.parse(original_code)  # Parse the original code into an abstract syntax tree
            for node in ast.walk(tree):  # Walk through each node in the AST
                # Check for dangerous function calls
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        if node.func.id in ['eval', 'exec', 'os.system']:
                            issues.append(SecurityIssue(
                                severity="High",
                                description=f"Dangerous function '{node.func.id}' detected",
                                line_number=node.lineno,
                                suggestion=f"Avoid using {node.func.id}. Consider using safer alternatives.",
                                code_snippet=original_code.split('\n')[node.lineno-1]
                            ))

                # Check for hardcoded secrets
                if isinstance(node, ast.Assign):
                    for target in node.targets:
                        if isinstance(target, ast.Name):
                            if any(secret in target.id.lower() for secret in ['password', 'secret', 'key', 'token']):
                                issues.append(SecurityIssue(
                                    severity="High",
                                    description="Potential hardcoded secret detected",
                                    line_number=node.lineno,
                                    suggestion="Use environment variables for sensitive data",
                                    code_snippet=original_code.split('\n')[node.lineno-1]
                                ))
        except SyntaxError:
            # Handle syntax errors in the original code
            issues.append(SecurityIssue(
                severity="Error",
                description="Invalid Python syntax",
                line_number=0,
                suggestion="Please fix syntax errors first",
                code_snippet=""
            ))

        return issues  # Return the list of identified security issues

In [74]:
def create_gradio_interface(analyzer: CodeSecurityAnalyzer):
    """Creates an enhanced Gradio interface for analyzing Python code security."""

    async def analyze_callback(code: str) -> str:
        """Callback function to analyze the provided code snippet."""
        # Check if the input code is empty
        if not code.strip():
            return "Please enter some code to analyze."

        # Analyze the provided code snippet for security issues
        issues = await analyzer.analyze_code_snippet(code)

        # If no issues are found, return a success message
        if not issues:
            return "✅ No security issues found."

        # Prepare the results for display
        result = "🔍 Security Analysis Results:\n\n"
        for issue in issues:
            # Assign a color-coded severity icon based on the issue's severity level
            severity_color = {
                "High": "🔴",
                "Medium": "🟡",
                "Low": "🟢",
                "Error": "⚠️"
            }.get(issue.severity, "⚪")

            # Format the output for each issue
            result += f"{'═'*50}\n"
            result += f"{severity_color} SEVERITY: {issue.severity}\n"
            result += f"📍 LINE {issue.line_number}: {issue.description}\n"
            if issue.code_snippet:
                result += f"💻 CODE:\n```python\n{issue.code_snippet}\n```\n"
            result += f"💡 SUGGESTION: {issue.suggestion}\n\n"

        return result

    # Example code snippets that may contain vulnerabilities
    examples = [
        ['''
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return execute_query(query)
        '''],
        ['''
@app.route('/profile')
def profile():
    template = request.args.get('template')
    return render_template_string(template)
        '''],
        ['''
def process_data(user_input):
    api_key = "secret_key_12345"
    exec(user_input)
    return {"status": "processed"}
        ''']
    ]

    # Custom CSS for styling the Gradio interface
    custom_css = """
    body {
        font-family: 'Arial', sans-serif;
        color: #e5e7eb;
    }
    .gradio-container {
        background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
        padding: 20px;
        border-radius: 10px;
    }
    .dark .gradio-container {
        background: linear-gradient(135deg, #0d1117 0%, #161b22 100%);
    }
    .light .gradio-container {
        background: linear-gradient(135deg, #f0f2f5 0%, #e2e8f0 100%);
    }
    .container {
        max-width: 1200px !important;
        margin: auto !important;
    }
    .gr-button {
        background: linear-gradient(90deg, #3b82f6 0%, #2563eb 100%) !important;
        border: none !important;
        color: white !important;
        padding: 10px 20px;
        border-radius: 5px;
        font-size: 16px;
    }
    .gr-button:hover {
        background: linear-gradient(90deg, #2563eb 0%, #1d4ed8 100%) !important;
        transform: translateY(-2px);
        box-shadow: 0 5px 15px rgba(37, 99, 235, 0.2);
    }
    .gr-box {
        border-radius: 12px !important;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1), 0 2px 4px -1px rgba(0, 0, 0, 0.06) !important;
        background-color: rgba(255, 255, 255, 0.1);
    }
    .dark .gr-box {
        background-color: rgba(31, 41, 55, 0.8) !important;
    }
    .light .gr-box {
        background-color: rgba(255, 255, 255, 0.9) !important;
    }
    code {
        background-color: #1f2937 !important;
        color: #e5e7eb !important;
        padding: 0. 2em 0.4em !important;
        border-radius: 4px !important;
    }
    h3 {
        color: #ffffff;
    }
    p {
        color: #d1d5db;
    }
    """

    # Create the Gradio interface with the specified settings
    interface = gr.Interface(
        fn=analyze_callback,
        inputs=gr.Code(
            language="python",
            label="Enter Python Code",
            lines=10,
            container=False,
        ),
        outputs=gr.Textbox(
            label="Security Analysis",
            lines=12,
            container=False
        ),
        title="🛡️ Dyxa Guardian",
        description="""
        <div style='text-align: center; margin-bottom: 1rem'>
            <h3>Real-time Security Analysis powered by Cerebras API</h3>
            <p>Analyze your Python code for security vulnerabilities in real-time.</p>
        </div>
        """,
        examples=examples,
        live=True,
        css=custom_css,
        theme=gr.themes.Soft(
            primary_hue="blue",
            secondary_hue="blue",
            neutral_hue="slate",
        ),
        allow_flagging="never"
    )

    return interface

In [75]:
# Create an instance of the CodeSecurityAnalyzer class
analyzer = CodeSecurityAnalyzer()

# Create a Gradio interface for the analyzer
interface = create_gradio_interface(analyzer)

# Launch the Gradio interface with debugging enabled and sharing options
interface.launch(debug=True, share=True)

/usr/local/lib/python3.10/dist-packages/gradio/interface.py:399: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://48691fce4932d0679d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://48691fce4932d0679d.gradio.live
